# Kerala Pipeline 1 Preprocessing: Sentinel-1 + Sentinel-2 + DEM

Colab-ready preprocessing notebook for Pipeline 1.

It prepares model-ready patch chunks for a Siamese U-Net flood-mask workflow:

```text
Sentinel-1 pre/flood + Sentinel-2 pre/flood + DEM -> patch tensors + flood mask patches
```

Outputs are saved to Google Drive under `Kerala_Pipeline1_Model_Ready` by default.


## 1. Install Dependencies

Run this in Colab. `rasterio` is required for GeoTIFF reading and reprojection.


In [ ]:
!pip install rasterio tqdm -q


## 2. Mount Drive and Imports


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import calendar
import gc
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling as RioResampling
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')


## 3. Configure Google Drive Paths

Update these paths if your Drive folder names differ.

Expected default layout:

```text
/content/drive/MyDrive/PRJ3_Data/KERELA_PRJ3/
  S1_KERELA_2019_STACK/
  S1_KERELA_2020_STACK/
  ...
  S2_KERELA_2019_STACK/
  S2_KERELA_2020_STACK/
  ...
  DEM_KERELA/
```

The notebook also tries to auto-detect Sentinel-2 folders if exact names differ.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/PRJ3_DATA')
if not DRIVE_ROOT.exists():
    DRIVE_ROOT = Path('/content/drive/MyDrive/Prj_3_Data')

DATA_ROOT = DRIVE_ROOT / 'KERELA_PRJ3'
OUTPUT_ROOT = DRIVE_ROOT / 'Kerala_Pipeline1_Model_Ready'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

YEARS = [2018, 2019, 2020, 2021, 2022]

S1_EVENT_FOLDERS = {
    2018: DATA_ROOT / 'S1_KERELA_2018_STACK',
    2019: DATA_ROOT / 'S1_KERELA_2019_STACK',
    2020: DATA_ROOT / 'S1_KERELA_2020_STACK',
    2021: DATA_ROOT / 'S1_KERELA_2021_STACK',
    2022: DATA_ROOT / 'S1_KERELA_2022_STACK',
}

# Edit these if your Sentinel-2 folders use different names.
S2_EVENT_FOLDERS = {
    2018: DATA_ROOT / 'S2_KERELA_2018_STACK',
    2019: DATA_ROOT / 'S2_KERELA_2019_STACK',
    2020: DATA_ROOT / 'S2_KERELA_2020_STACK',
    2021: DATA_ROOT / 'S2_KERELA_2021_STACK',
    2022: DATA_ROOT / 'S2_KERELA_2022_STACK',
}

DEM_FILE = DATA_ROOT / 'DEM_KERELA'

PATCH_SIZE = 128
STRIDE = 128
CHUNK_SIZE = 1500
MIN_FLOOD_FRACTION = 0.01
KEEP_EVERY_N_EMPTY = 5

print('DRIVE_ROOT:', DRIVE_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


## 4. Event Windows and File Discovery

Files are classified as `pre`, `flood`, or `post` using filename tags first. If tags are absent, the notebook parses window names such as `July_1_to_July_15` and compares them with Kerala flood-event windows.


In [ ]:
EVENT_DATES = {
    2018: ('2018-06-01', '2018-08-19'),
    2019: ('2019-08-01', '2019-08-31'),
    2020: ('2020-06-01', '2020-08-18'),
    2021: ('2021-10-11', '2021-10-26'),
    2022: ('2022-08-01', '2022-09-10'),
}

MONTH_TO_NUMBER = {month.lower(): idx for idx, month in enumerate(calendar.month_name) if month}


def find_tifs(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(folder.rglob('*.tif')) + sorted(folder.rglob('*.tiff'))


def auto_find_sensor_folder(year, sensor_prefix):
    exact = DATA_ROOT / f'{sensor_prefix}_KERELA_{year}_STACK'
    if exact.exists():
        return exact
    patterns = [
        f'*{sensor_prefix}*KERELA*{year}*',
        f'*KERELA*{sensor_prefix}*{year}*',
        f'*{sensor_prefix.lower()}*kerela*{year}*',
        f'*{sensor_prefix}*KERALA*{year}*',
        f'*KERALA*{sensor_prefix}*{year}*',
        f'*{sensor_prefix.lower()}*kerala*{year}*',
    ]
    for pattern in patterns:
        matches = sorted(DATA_ROOT.glob(pattern))
        matches = [p for p in matches if p.is_dir()]
        if matches:
            return matches[0]
    return exact

for year in YEARS:
    if not S2_EVENT_FOLDERS[year].exists():
        S2_EVENT_FOLDERS[year] = auto_find_sensor_folder(year, 'S2')


def parse_window_dates(path, year):
    match = re.search(r'([A-Za-z]+)_(\d{1,2})_to_([A-Za-z]+)_(\d{1,2})', path.stem)
    if not match:
        return None, None
    start_month, start_day, end_month, end_day = match.groups()
    start_month_num = MONTH_TO_NUMBER.get(start_month.lower())
    end_month_num = MONTH_TO_NUMBER.get(end_month.lower())
    if start_month_num is None or end_month_num is None:
        return None, None
    start_year = year
    end_year = year + 1 if end_month_num < start_month_num else year
    start_date = np.datetime64(f'{start_year}-{start_month_num:02d}-{int(start_day):02d}')
    end_date = np.datetime64(f'{end_year}-{end_month_num:02d}-{int(end_day):02d}')
    return start_date, end_date


def classify_file(path, year=None):
    name = path.name.upper()
    if 'PRE' in name:
        return 'pre'
    if 'FLOOD' in name and 'WINDOW' not in name:
        return 'flood'
    if 'POST' in name:
        return 'post'
    if year is None:
        return 'window'
    start_date, end_date = parse_window_dates(path, year)
    if start_date is None:
        return 'unknown'
    event_start = np.datetime64(EVENT_DATES[year][0])
    event_end = np.datetime64(EVENT_DATES[year][1])
    if end_date <= event_start:
        return 'pre'
    if start_date >= event_end:
        return 'post'
    if start_date < event_end and end_date > event_start:
        return 'flood'
    return 'window'


def window_event_overlap_days(path, year):
    start_date, end_date = parse_window_dates(path, year)
    if start_date is None:
        return 0
    event_start = np.datetime64(EVENT_DATES[year][0])
    event_end = np.datetime64(EVENT_DATES[year][1])
    overlap_start = max(start_date, event_start)
    overlap_end = min(end_date, event_end)
    if overlap_end <= overlap_start:
        return 0
    return int((overlap_end - overlap_start).astype('timedelta64[D]').astype(int))


def days_from_event(path, year, side):
    start_date, end_date = parse_window_dates(path, year)
    if start_date is None:
        return 10**9
    event_start = np.datetime64(EVENT_DATES[year][0])
    event_end = np.datetime64(EVENT_DATES[year][1])
    if side == 'pre':
        return abs(int((event_start - end_date).astype('timedelta64[D]').astype(int)))
    if side == 'post':
        return abs(int((start_date - event_end).astype('timedelta64[D]').astype(int)))
    return 10**9


def group_sensor_files(sensor_folders):
    grouped = {}
    for year, folder in sensor_folders.items():
        files = find_tifs(folder)
        grouped[year] = {'pre': [], 'flood': [], 'post': [], 'window': [], 'unknown': []}
        for path in files:
            grouped[year][classify_file(path, year)].append(path)
    return grouped


def pick_representative_windows(year, groups):
    selected = {'pre': None, 'flood': None, 'post': None}
    if groups['pre']:
        selected['pre'] = min(groups['pre'], key=lambda p: days_from_event(p, year, 'pre'))
    if groups['flood']:
        selected['flood'] = max(groups['flood'], key=lambda p: window_event_overlap_days(p, year))
    if groups['post']:
        selected['post'] = min(groups['post'], key=lambda p: days_from_event(p, year, 'post'))
    return selected

s1_grouped = group_sensor_files(S1_EVENT_FOLDERS)
s2_grouped = group_sensor_files(S2_EVENT_FOLDERS)

for sensor_name, grouped in [('S1', s1_grouped), ('S2', s2_grouped)]:
    print('\n', sensor_name)
    for year, groups in grouped.items():
        selected = pick_representative_windows(year, groups)
        print(year, 'folder:', (S1_EVENT_FOLDERS if sensor_name == 'S1' else S2_EVENT_FOLDERS)[year])
        print(' counts:', {k: len(v) for k, v in groups.items()})
        print(' selected:', {k: (v.name if v else None) for k, v in selected.items()})

# Diagnostics only. The old working notebook continued from here and later saved arrays
# from the grouped stage lists, so we do not hard-stop at discovery time.
discovery_rows = []
for year in YEARS:
    selected = pick_representative_windows(year, s1_grouped[year])
    discovery_rows.append({
        'year': year,
        's1_folder': str(S1_EVENT_FOLDERS[year]),
        's1_folder_exists': S1_EVENT_FOLDERS[year].exists(),
        's1_tif_count': sum(len(v) for v in s1_grouped[year].values()),
        's1_pre_count': len(s1_grouped[year]['pre']),
        's1_flood_count': len(s1_grouped[year]['flood']),
        'selected_pre': None if selected['pre'] is None else selected['pre'].name,
        'selected_flood': None if selected['flood'] is None else selected['flood'].name,
        's2_folder': str(S2_EVENT_FOLDERS[year]),
        's2_folder_exists': S2_EVENT_FOLDERS[year].exists(),
        's2_tif_count': sum(len(v) for v in s2_grouped[year].values()),
    })

discovery_df = pd.DataFrame(discovery_rows)
display(discovery_df)
print('If pre/flood counts are 0, edit S1_EVENT_FOLDERS or filename classification before continuing.')


## 5. Raster Preprocessing Functions

Sentinel-1 is normalized as SAR backscatter channels. Sentinel-2 is normalized as reflectance bands and converted into spectral indices.

The notebook assumes Sentinel-2 GeoTIFFs are either multi-band stacks or single-band stacks. If your band order differs, edit `S2_BAND_ORDER`.


In [ ]:
def normalize_band(band, min_val, max_val):
    band = np.nan_to_num(band, nan=min_val, posinf=max_val, neginf=min_val)
    band = np.clip(band, min_val, max_val)
    denom = max(max_val - min_val, 1e-6)
    return ((band - min_val) / denom).astype(np.float32)


def safe_divide(a, b):
    return (a - b) / (a + b + 1e-6)


def preprocess_s1_tif(path):
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)
        profile = src.profile.copy()
    if img.shape[0] < 2:
        raise ValueError(f'Sentinel-1 file needs at least VV/VH bands: {path}')
    img = np.nan_to_num(img, nan=-30.0)
    vv = normalize_band(img[0], -30, 5)
    vh = normalize_band(img[1], -35, 0)
    stack = np.stack([vv, vh, vv - vh, vv + vh], axis=0)
    return stack.astype(np.float32), profile


def estimate_valid_coverage(path, band_indexes=None, preview_size=512):
    """Estimate valid raster coverage from a small preview read, not the full scene."""
    with rasterio.open(path) as src:
        if band_indexes is None:
            band_indexes = list(range(1, min(src.count, 4) + 1))
        band_indexes = [idx for idx in band_indexes if idx <= src.count]
        if not band_indexes:
            return 0.0
        scale = max(src.width, src.height) / preview_size
        out_h = max(1, int(src.height / max(scale, 1)))
        out_w = max(1, int(src.width / max(scale, 1)))
        arr = src.read(band_indexes, out_shape=(len(band_indexes), out_h, out_w), masked=True)
    data = np.ma.filled(arr, 0).astype(np.float32, copy=False)
    valid = np.any(np.isfinite(data) & (np.abs(data) > 1e-6), axis=0)
    return float(valid.mean())


def pick_best_coverage_window(files, fallback=None, sensor='s1'):
    files = [p for p in files if p is not None]
    if not files:
        return fallback, 0.0
    band_indexes = [1, 2] if sensor.lower() == 's1' else [1, 2, 3, 4]
    scored = []
    for path in files:
        try:
            scored.append((estimate_valid_coverage(path, band_indexes=band_indexes), path))
        except Exception as exc:
            print(f'coverage check failed for {path.name}: {exc}')
    if not scored:
        return fallback, 0.0
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[0][1], scored[0][0]

# Default Sentinel-2 band positions for a multi-band stack.
# Edit this if your stack has a different order.
S2_BAND_ORDER = {
    'blue': 0,
    'green': 1,
    'red': 2,
    'nir': 3,
    'swir1': 4,
    'swir2': 5,
}

# Full-scene S1/S2 arrays are very large. Keep SAVE_ALL_* False on Colab.
# Composite mode is too heavy for standard Colab, so the safe default selects
# one best-coverage window per sensor/stage/year.
USE_STAGE_COMPOSITES = False
SELECT_BEST_COVERAGE_WINDOW = True
SAVE_ALL_S1_WINDOWS = False
S1_STAGES_TO_SAVE = ['pre', 'flood', 'post']
SAVE_ALL_S2_WINDOWS = False
S2_STAGES_TO_SAVE = ['pre', 'flood', 'post']

# Resume support: rerunning Cell 6 will skip valid outputs already written to Drive.
SKIP_EXISTING_OUTPUTS = True


def scale_s2_reflectance(band):
    band = band.astype(np.float32)
    finite = band[np.isfinite(band)]
    if finite.size and np.nanmax(finite) > 2.0:
        band = band / 10000.0
    band = np.nan_to_num(band, nan=0.0, posinf=1.0, neginf=0.0)
    return np.clip(band, 0.0, 1.0).astype(np.float32)


def preprocess_s2_tif(path):
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)
        profile = src.profile.copy()
    if img.shape[0] < 4:
        raise ValueError(f'Sentinel-2 file needs at least Blue/Green/Red/NIR bands: {path}')

    def get_band(name, fallback=None):
        idx = S2_BAND_ORDER.get(name)
        if idx is not None and idx < img.shape[0]:
            return scale_s2_reflectance(img[idx])
        if fallback is not None:
            return fallback
        return np.zeros(img.shape[1:], dtype=np.float32)

    blue = get_band('blue')
    green = get_band('green')
    red = get_band('red')
    nir = get_band('nir')
    swir1 = get_band('swir1', fallback=np.zeros_like(nir))
    swir2 = get_band('swir2', fallback=np.zeros_like(nir))

    ndvi = safe_divide(nir, red).astype(np.float32)
    ndwi = safe_divide(green, nir).astype(np.float32)
    mndwi = safe_divide(green, swir1).astype(np.float32)
    ndbi = safe_divide(swir1, nir).astype(np.float32)

    # Keep raw reflectance bands plus water/vegetation indices.
    stack = np.stack([blue, green, red, nir, swir1, swir2, ndvi, ndwi, mndwi, ndbi], axis=0)
    stack = np.nan_to_num(stack, nan=0.0, posinf=1.0, neginf=-1.0)
    return stack.astype(np.float32), profile


def find_dem_tif(dem_path):
    dem_path = Path(dem_path)
    if dem_path.is_file():
        return dem_path
    tifs = sorted(dem_path.glob('*.tif')) + sorted(dem_path.glob('*.tiff'))
    if not tifs:
        raise FileNotFoundError(f'No DEM GeoTIFF found under {dem_path}')
    return tifs[0]


def preprocess_dem_full(dem_path, target_profile):
    dem_tif = find_dem_tif(dem_path)
    dem_data = np.empty((target_profile['height'], target_profile['width']), dtype=np.float32)
    with rasterio.open(dem_tif) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dem_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_profile['transform'],
            dst_crs=target_profile['crs'],
            resampling=Resampling.bilinear,
        )
    dem_data = np.nan_to_num(dem_data, nan=0.0)
    dem_data = np.clip(dem_data, 0, 2700) / 2700
    return dem_data.astype(np.float32)


def resample_array_to_profile(path, preprocess_fn, target_profile, resampling=Resampling.bilinear):
    src_stack, src_profile = preprocess_fn(path)
    dst = np.zeros((src_stack.shape[0], target_profile['height'], target_profile['width']), dtype=np.float32)
    with rasterio.open(path) as src:
        for band_idx in range(src_stack.shape[0]):
            reproject(
                source=src_stack[band_idx],
                destination=dst[band_idx],
                src_transform=src_profile['transform'],
                src_crs=src_profile['crs'],
                dst_transform=target_profile['transform'],
                dst_crs=target_profile['crs'],
                resampling=resampling,
            )
    return dst.astype(np.float32)


S2_OUTPUT_CHANNELS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'ndwi', 'mndwi', 'ndbi']


def _scale_reflectance_inplace(arr):
    arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0, copy=False)
    if np.nanmax(arr) > 2.0:
        arr /= 10000.0
    np.clip(arr, 0.0, 1.0, out=arr)
    return arr


def _compute_index_chunked(mm, out_ch, a_ch, b_ch, chunk_rows=256):
    height = mm.shape[1]
    for row0 in range(0, height, chunk_rows):
        row1 = min(row0 + chunk_rows, height)
        a = mm[a_ch, row0:row1]
        b = mm[b_ch, row0:row1]
        numerator = np.empty_like(a, dtype=np.float32)
        denominator = np.empty_like(a, dtype=np.float32)
        np.subtract(a, b, out=numerator)
        np.add(a, b, out=denominator)
        denominator += 1e-6
        np.divide(numerator, denominator, out=numerator)
        np.nan_to_num(numerator, nan=0.0, posinf=1.0, neginf=-1.0, copy=False)
        mm[out_ch, row0:row1] = numerator


def save_s2_aligned_memmap(path, out_path, target_profile, resampling=Resampling.bilinear):
    """Save aligned Sentinel-2 features without holding the full 10-channel scene in RAM."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shape = (len(S2_OUTPUT_CHANNELS), target_profile['height'], target_profile['width'])
    mm = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float32, shape=shape)
    mm[:] = 0.0

    with rasterio.open(path) as src:
        if src.count < 4:
            raise ValueError(f'Sentinel-2 file needs at least Blue/Green/Red/NIR bands: {path}')
        for name, out_ch in [('blue', 0), ('green', 1), ('red', 2), ('nir', 3), ('swir1', 4), ('swir2', 5)]:
            idx = S2_BAND_ORDER.get(name)
            if idx is None or idx >= src.count:
                continue
            reproject(
                source=rasterio.band(src, idx + 1),
                destination=mm[out_ch],
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=src.nodata,
                dst_transform=target_profile['transform'],
                dst_crs=target_profile['crs'],
                dst_nodata=0.0,
                resampling=resampling,
            )
            _scale_reflectance_inplace(mm[out_ch])
            mm.flush()

    _compute_index_chunked(mm, 6, 3, 2)  # NDVI: NIR, Red
    _compute_index_chunked(mm, 7, 1, 3)  # NDWI: Green, NIR
    _compute_index_chunked(mm, 8, 1, 4)  # MNDWI: Green, SWIR1
    _compute_index_chunked(mm, 9, 4, 3)  # NDBI: SWIR1, NIR
    mm.flush()
    shape = mm.shape
    del mm
    gc.collect()
    return shape


def _update_mean_from_chunked(dest, count, new_arr, valid_eps=1e-6, chunk_rows=256):
    height = new_arr.shape[-2]
    for row0 in range(0, height, chunk_rows):
        row1 = min(row0 + chunk_rows, height)
        chunk = new_arr[:, row0:row1]
        valid = np.any(np.isfinite(chunk) & (np.abs(chunk) > valid_eps), axis=0)
        if not valid.any():
            continue
        count_chunk = count[row0:row1]
        new_count = count_chunk[valid] + 1
        for ch in range(dest.shape[0]):
            dest_chunk = dest[ch, row0:row1]
            current = dest_chunk[valid]
            current += (chunk[ch][valid] - current) / new_count
            dest_chunk[valid] = current
        count_chunk[valid] = new_count


def _update_mean_channel_from_chunked(dest, count, new_arr, valid_eps=1e-6, chunk_rows=256):
    height = new_arr.shape[0]
    for row0 in range(0, height, chunk_rows):
        row1 = min(row0 + chunk_rows, height)
        chunk = new_arr[row0:row1]
        valid = np.isfinite(chunk) & (np.abs(chunk) > valid_eps)
        if not valid.any():
            continue
        count_chunk = count[row0:row1]
        dest_chunk = dest[row0:row1]
        new_count = count_chunk[valid] + 1
        current = dest_chunk[valid]
        current += (chunk[valid] - current) / new_count
        dest_chunk[valid] = current
        count_chunk[valid] = new_count


def save_s1_composite(paths, out_path):
    """Save one S1 stage composite by averaging valid pixels from all stage windows."""
    paths = list(paths)
    if not paths:
        raise ValueError('No S1 files supplied for composite')
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    first_arr, profile = preprocess_s1_tif(paths[0])
    shape = first_arr.shape
    del first_arr
    gc.collect()
    mm = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float32, shape=shape)
    mm[:] = 0.0
    count_path = out_path.with_name(f'_{out_path.stem}_valid_count.npy')
    count = np.lib.format.open_memmap(count_path, mode='w+', dtype=np.uint16, shape=shape[1:])
    count[:] = 0

    used = 0
    try:
        for tif in tqdm(paths, desc=f'composite {out_path.stem}', leave=False):
            arr, arr_profile = preprocess_s1_tif(tif)
            if arr.shape != shape:
                print(f'skipping {tif.name}: shape {arr.shape} != {shape}')
                continue
            _update_mean_from_chunked(mm, count, arr)
            used += 1
            del arr
            gc.collect()
        mm.flush()
        profile = profile.copy()
        shape = mm.shape
    finally:
        del count
        del mm
        gc.collect()
        try:
            count_path.unlink()
        except FileNotFoundError:
            pass
    if used == 0:
        raise ValueError(f'No S1 files could be composited for {out_path.name}')
    return shape, profile, used


def save_s2_composite_memmap(paths, out_path, target_profile, resampling=Resampling.bilinear):
    """Save one aligned S2 stage composite from all stage windows without full-scene RAM spikes."""
    paths = list(paths)
    if not paths:
        raise ValueError('No S2 files supplied for composite')
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shape = (len(S2_OUTPUT_CHANNELS), target_profile['height'], target_profile['width'])
    mm = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float32, shape=shape)
    mm[:] = 0.0
    tmp_path = out_path.with_name(f'_{out_path.stem}_tmp_band.npy')
    count_path = out_path.with_name(f'_{out_path.stem}_valid_count.npy')
    tmp = np.lib.format.open_memmap(tmp_path, mode='w+', dtype=np.float32, shape=shape[1:])
    count = np.lib.format.open_memmap(count_path, mode='w+', dtype=np.uint16, shape=shape[1:])

    try:
        for name, out_ch in [('blue', 0), ('green', 1), ('red', 2), ('nir', 3), ('swir1', 4), ('swir2', 5)]:
            count[:] = 0
            band_used = 0
            for tif in tqdm(paths, desc=f'composite {out_path.stem} {name}', leave=False):
                with rasterio.open(tif) as src:
                    idx = S2_BAND_ORDER.get(name)
                    if idx is None or idx >= src.count:
                        continue
                    tmp[:] = 0.0
                    reproject(
                        source=rasterio.band(src, idx + 1),
                        destination=tmp,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        src_nodata=src.nodata,
                        dst_transform=target_profile['transform'],
                        dst_crs=target_profile['crs'],
                        dst_nodata=0.0,
                        resampling=resampling,
                    )
                _scale_reflectance_inplace(tmp)
                _update_mean_channel_from_chunked(mm[out_ch], count, tmp)
                band_used += 1
                tmp.flush()
            print(f'{out_path.stem}: {name} composited from {band_used} files')

        _compute_index_chunked(mm, 6, 3, 2)  # NDVI: NIR, Red
        _compute_index_chunked(mm, 7, 1, 3)  # NDWI: Green, NIR
        _compute_index_chunked(mm, 8, 1, 4)  # MNDWI: Green, SWIR1
        _compute_index_chunked(mm, 9, 4, 3)  # NDBI: SWIR1, NIR
        mm.flush()
        shape = mm.shape
    finally:
        del tmp
        del count
        del mm
        gc.collect()
        for temp_path in [tmp_path, count_path]:
            try:
                temp_path.unlink()
            except FileNotFoundError:
                pass
    return shape, len(paths)


def save_numpy_array(arr, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, arr)


def valid_npy(path, expected_ndim=None):
    path = Path(path)
    if not path.exists():
        return False, None
    try:
        arr = np.load(path, mmap_mode='r')
        shape = arr.shape
        if expected_ndim is not None and arr.ndim != expected_ndim:
            del arr
            return False, shape
        # Touch one value so truncated memmaps fail immediately.
        _ = arr.reshape(-1)[0]
        del arr
        return True, shape
    except Exception as exc:
        print(f'{path.name} exists but is not readable/complete: {exc}')
        return False, None


## 6. Save Aligned Full-Year Arrays

For each year, this cell saves representative pre/flood arrays for S1 and S2, aligned to the S1 grid. DEM is also aligned to the S1 grid.


In [ ]:
array_index = []

for year in YEARS:
    print(f'\n=== Processing {year} ===')
    year_out = OUTPUT_ROOT / str(year)
    year_out.mkdir(parents=True, exist_ok=True)

    ref_profile = None
    saved_s1 = 0

    # Save one S1 output per stage. In composite mode, each output is filled from all windows in that stage.
    selected_s1 = pick_representative_windows(year, s1_grouped[year])
    for stage in S1_STAGES_TO_SAVE:
        if SAVE_ALL_S1_WINDOWS:
            s1_files = s1_grouped[year][stage]
            for tif in tqdm(s1_files, desc=f'{year} S1 {stage}'):
                try:
                    arr, profile = preprocess_s1_tif(tif)
                    if ref_profile is None:
                        ref_profile = profile
                    out_path = year_out / f's1_{stage}_{tif.stem}.npy'
                    save_numpy_array(arr, out_path)
                    array_index.append({
                        'year': year,
                        'sensor': 's1',
                        'stage': stage,
                        'source': str(tif),
                        'array': str(out_path),
                        'shape': str(arr.shape),
                    })
                    saved_s1 += 1
                    print('saved', out_path.name, arr.shape)
                    del arr
                    gc.collect()
                except Exception as exc:
                    print(f'{year}: skipping S1 {stage} {tif.name}: {exc}')
        else:
            if USE_STAGE_COMPOSITES:
                s1_files = s1_grouped[year][stage]
            elif SELECT_BEST_COVERAGE_WINDOW:
                best_s1, coverage = pick_best_coverage_window(s1_grouped[year][stage], fallback=selected_s1[stage], sensor='s1')
                s1_files = [best_s1] if best_s1 else []
                if best_s1:
                    print(f'{year} S1 {stage}: selected {best_s1.name} coverage~{coverage:.3f}')
            else:
                s1_files = [selected_s1[stage]] if selected_s1[stage] else []
            if not s1_files:
                print(f'{year}: no S1 {stage} files')
                continue
            try:
                out_path = year_out / f's1_{stage}_composite.npy' if USE_STAGE_COMPOSITES else year_out / f's1_{stage}_best_coverage.npy'
                existing_ok, existing_shape = valid_npy(out_path, expected_ndim=3)
                if SKIP_EXISTING_OUTPUTS and not existing_ok and not USE_STAGE_COMPOSITES:
                    alternate_path = year_out / f's1_{stage}_composite.npy'
                    alternate_ok, alternate_shape = valid_npy(alternate_path, expected_ndim=3)
                    if alternate_ok:
                        out_path = alternate_path
                        existing_ok, existing_shape = alternate_ok, alternate_shape
                if SKIP_EXISTING_OUTPUTS and existing_ok:
                    if ref_profile is None:
                        with rasterio.open(s1_files[0]) as src:
                            ref_profile = src.profile.copy()
                    source_count = len(s1_files) if USE_STAGE_COMPOSITES else 1
                    array_index.append({
                        'year': year,
                        'sensor': 's1',
                        'stage': stage,
                        'source': json.dumps([str(p) for p in s1_files]),
                        'source_count': source_count,
                        'array': str(out_path),
                        'shape': str(existing_shape),
                    })
                    saved_s1 += 1
                    print('skip existing', out_path.name, existing_shape)
                    continue
                if USE_STAGE_COMPOSITES:
                    shape, profile, source_count = save_s1_composite(s1_files, out_path)
                else:
                    arr, profile = preprocess_s1_tif(s1_files[0])
                    save_numpy_array(arr, out_path)
                    shape = arr.shape
                    source_count = 1
                    del arr
                    gc.collect()
                if ref_profile is None:
                    ref_profile = profile
                array_index.append({
                    'year': year,
                    'sensor': 's1',
                    'stage': stage,
                    'source': json.dumps([str(p) for p in s1_files]),
                    'source_count': source_count,
                    'array': str(out_path),
                    'shape': str(shape),
                })
                saved_s1 += 1
                print('saved', out_path.name, shape, 'sources:', source_count)
                gc.collect()
            except Exception as exc:
                print(f'{year}: skipping S1 {stage} composite: {exc}')

    if ref_profile is None:
        print(f'{year}: no S1 arrays saved; cannot align DEM/S2 for this year')
        continue

    # Optional S2 support. In composite mode, each stage output is filled from all S2 windows in that stage.
    saved_s2 = 0
    selected_s2 = pick_representative_windows(year, s2_grouped[year])
    for stage in S2_STAGES_TO_SAVE:
        if SAVE_ALL_S2_WINDOWS:
            s2_files = s2_grouped[year][stage]
            for tif in tqdm(s2_files, desc=f'{year} S2 {stage}'):
                try:
                    out_path = year_out / f's2_{stage}_{tif.stem}.npy'
                    shape = save_s2_aligned_memmap(tif, out_path, ref_profile, resampling=Resampling.bilinear)
                    array_index.append({
                        'year': year,
                        'sensor': 's2',
                        'stage': stage,
                        'source': str(tif),
                        'source_count': 1,
                        'array': str(out_path),
                        'shape': str(shape),
                    })
                    saved_s2 += 1
                    print('saved', out_path.name, shape)
                    gc.collect()
                except Exception as exc:
                    print(f'{year}: skipping S2 {stage} {tif.name}: {exc}')
        else:
            if USE_STAGE_COMPOSITES:
                s2_files = s2_grouped[year][stage]
            elif SELECT_BEST_COVERAGE_WINDOW:
                best_s2, coverage = pick_best_coverage_window(s2_grouped[year][stage], fallback=selected_s2[stage], sensor='s2')
                s2_files = [best_s2] if best_s2 else []
                if best_s2:
                    print(f'{year} S2 {stage}: selected {best_s2.name} coverage~{coverage:.3f}')
            else:
                s2_files = [selected_s2[stage]] if selected_s2[stage] else []
            if not s2_files:
                print(f'{year}: no S2 {stage} files')
                continue
            try:
                out_path = year_out / f's2_{stage}_composite.npy' if USE_STAGE_COMPOSITES else year_out / f's2_{stage}_best_coverage.npy'
                existing_ok, existing_shape = valid_npy(out_path, expected_ndim=3)
                if SKIP_EXISTING_OUTPUTS and not existing_ok and not USE_STAGE_COMPOSITES:
                    alternate_path = year_out / f's2_{stage}_composite.npy'
                    alternate_ok, alternate_shape = valid_npy(alternate_path, expected_ndim=3)
                    if alternate_ok:
                        out_path = alternate_path
                        existing_ok, existing_shape = alternate_ok, alternate_shape
                if SKIP_EXISTING_OUTPUTS and existing_ok:
                    source_count = len(s2_files) if USE_STAGE_COMPOSITES else 1
                    array_index.append({
                        'year': year,
                        'sensor': 's2',
                        'stage': stage,
                        'source': json.dumps([str(p) for p in s2_files]),
                        'source_count': source_count,
                        'array': str(out_path),
                        'shape': str(existing_shape),
                    })
                    saved_s2 += 1
                    print('skip existing', out_path.name, existing_shape)
                    continue
                if USE_STAGE_COMPOSITES:
                    shape, source_count = save_s2_composite_memmap(s2_files, out_path, ref_profile, resampling=Resampling.bilinear)
                else:
                    shape = save_s2_aligned_memmap(s2_files[0], out_path, ref_profile, resampling=Resampling.bilinear)
                    source_count = 1
                array_index.append({
                    'year': year,
                    'sensor': 's2',
                    'stage': stage,
                    'source': json.dumps([str(p) for p in s2_files]),
                    'source_count': source_count,
                    'array': str(out_path),
                    'shape': str(shape),
                })
                saved_s2 += 1
                print('saved', out_path.name, shape, 'sources:', source_count)
                gc.collect()
            except Exception as exc:
                print(f'{year}: skipping S2 {stage} composite: {exc}')

    try:
        dem_path = year_out / 'DEM.npy'
        existing_ok, existing_shape = valid_npy(dem_path, expected_ndim=2)
        if SKIP_EXISTING_OUTPUTS and existing_ok:
            array_index.append({
                'year': year,
                'sensor': 'dem',
                'stage': 'static',
                'source': str(DEM_FILE),
                'array': str(dem_path),
                'shape': str(existing_shape),
            })
            print(f'{year}: DEM exists -> {existing_shape}')
            continue
        dem = preprocess_dem_full(DEM_FILE, ref_profile)
        save_numpy_array(dem, dem_path)
        array_index.append({
            'year': year,
            'sensor': 'dem',
            'stage': 'static',
            'source': str(DEM_FILE),
            'array': str(dem_path),
            'shape': str(dem.shape),
        })
        print(f'{year}: DEM saved -> {dem.shape}')
        del dem
        gc.collect()
    except Exception as exc:
        print(f'{year}: DEM failed: {exc}')

    print(f'{year}: saved S1 arrays={saved_s1}, saved S2 arrays={saved_s2}')

array_index_df = pd.DataFrame(array_index)
array_index_df.to_csv(OUTPUT_ROOT / 'array_index.csv', index=False)
print('\nArray index saved:', OUTPUT_ROOT / 'array_index.csv')
display(array_index_df.groupby(['year', 'sensor', 'stage']).size().reset_index(name='files') if not array_index_df.empty else array_index_df)


## 7. Create Quick Flood Masks

This creates pseudo flood masks from Sentinel-1 pre/flood change. If you already have validated masks, replace this cell with loading those masks into the same output filename pattern.


In [ ]:
def load_first_array(year, prefix, stage):
    year_dir = OUTPUT_ROOT / str(year)
    patterns = [f'{prefix}_{stage}_best_coverage.npy', f'{prefix}_{stage}_composite.npy', f'{prefix}_{stage}_*.npy']
    # Compatibility with the old Kerala_preprocessing output names.
    if prefix == 's1':
        patterns.append(f'{stage}_*.npy')
    files = []
    for pattern in patterns:
        for file in sorted(year_dir.glob(pattern)):
            if file not in files:
                files.append(file)
    if not files:
        return None, None
    for file in files:
        try:
            arr = np.load(file, mmap_mode='r')
            # Touch one value so empty/truncated files fail here instead of later cells.
            _ = arr.reshape(-1)[0]
            return arr, file
        except Exception as exc:
            print(f'skipping unreadable {file.name}: {exc}')
    return None, None


def make_quick_flood_mask_from_s1(pre_arr, flood_arr):
    pre_vv, pre_vh = pre_arr[0], pre_arr[1]
    flood_vv, flood_vh = flood_arr[0], flood_arr[1]
    delta_vv = flood_vv - pre_vv
    delta_vh = flood_vh - pre_vh
    mask = (
        ((delta_vh < -0.05) & (flood_vh < 0.52)) |
        ((delta_vv < -0.05) & (flood_vv < 0.55))
    )
    return mask.astype(np.uint8)

mask_index = []
for year in YEARS:
    pre_arr, pre_path = load_first_array(year, 's1', 'pre')
    flood_arr, flood_path = load_first_array(year, 's1', 'flood')
    if pre_arr is None or flood_arr is None:
        print(year, 'missing S1 arrays, skipping mask')
        continue
    mask = make_quick_flood_mask_from_s1(pre_arr, flood_arr)
    out_path = OUTPUT_ROOT / str(year) / f'Kerala_{year}_quick_flood_mask.npy'
    np.save(out_path, mask)
    mask_index.append({
        'year': year,
        'pre': str(pre_path),
        'flood': str(flood_path),
        'mask': str(out_path),
        'flood_pixel_fraction': float(mask.mean()),
    })
    print(year, 'mask saved:', out_path.name, 'flood fraction:', float(mask.mean()))
    del mask
    gc.collect()

pd.DataFrame(mask_index).to_csv(OUTPUT_ROOT / 'mask_index.csv', index=False)


## 8. Visual QA

Check one year before patch extraction. If masks look poor, adjust thresholds or replace quick masks with validated masks.


In [ ]:
def visualize_year(year, stride=12):
    s1_pre, _ = load_first_array(year, 's1', 'pre')
    s1_flood, _ = load_first_array(year, 's1', 'flood')
    s2_flood, _ = load_first_array(year, 's2', 'flood')
    mask_path = OUTPUT_ROOT / str(year) / f'Kerala_{year}_quick_flood_mask.npy'
    dem_path = OUTPUT_ROOT / str(year) / 'DEM.npy'
    if s1_pre is None or s1_flood is None or not mask_path.exists():
        print('Missing data for', year)
        return
    mask = np.load(mask_path, mmap_mode='r')
    dem = np.load(dem_path, mmap_mode='r') if dem_path.exists() else None
    delta_vh = s1_flood[1] - s1_pre[1]

    panels = [
        ('S1 PRE VH', s1_pre[1, ::stride, ::stride], 'gray'),
        ('S1 FLOOD VH', s1_flood[1, ::stride, ::stride], 'gray'),
        ('Delta VH', delta_vh[::stride, ::stride], 'RdBu'),
        ('Quick flood mask', mask[::stride, ::stride], 'Blues'),
    ]
    if s2_flood is not None:
        # Channel 8 is NDWI in preprocess_s2_tif.
        panels.append(('S2 FLOOD NDWI', s2_flood[7, ::stride, ::stride], 'BrBG'))
    if dem is not None:
        panels.append(('DEM', dem[::stride, ::stride], 'terrain'))

    plt.figure(figsize=(5 * len(panels), 5))
    for idx, (title, arr, cmap) in enumerate(panels, start=1):
        plt.subplot(1, len(panels), idx)
        plt.imshow(arr, cmap=cmap)
        plt.title(f'{year} {title}')
        plt.axis('off')
        plt.colorbar(fraction=0.046, pad=0.04)
    plt.show()

for year in YEARS:
    visualize_year(year, stride=12)


## 9. Extract Model-Ready Patch Chunks

Outputs per year:

```text
patches/X_s1_pre_*.npy
patches/X_s1_flood_*.npy
patches/X_s2_pre_*.npy
patches/X_s2_flood_*.npy
patches/X_dem_*.npy
patches/y_mask_*.npy
patches/patch_metadata_*.csv
```

These are ready for a multi-input Siamese U-Net.


In [ ]:
S2_CHANNELS = 10
S1_CHANNELS = 4

def zero_s1_patch():
    return np.zeros((S1_CHANNELS, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)

def zero_s2_patch():
    return np.zeros((S2_CHANNELS, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)

patch_index = []

for year in YEARS:
    print(f'\n=== Patch extraction {year} ===')
    s1_pre, _ = load_first_array(year, 's1', 'pre')
    s1_flood, _ = load_first_array(year, 's1', 'flood')
    s1_post, _ = load_first_array(year, 's1', 'post')
    s2_pre, _ = load_first_array(year, 's2', 'pre')
    s2_flood, _ = load_first_array(year, 's2', 'flood')
    s2_post, _ = load_first_array(year, 's2', 'post')
    mask_path = OUTPUT_ROOT / str(year) / f'Kerala_{year}_quick_flood_mask.npy'
    dem_path = OUTPUT_ROOT / str(year) / 'DEM.npy'

    missing_required = []
    if s1_pre is None:
        missing_required.append('s1_pre')
    if s1_flood is None:
        missing_required.append('s1_flood')
    if not mask_path.exists():
        missing_required.append('quick_flood_mask')
    if not dem_path.exists():
        missing_required.append('DEM')
    if missing_required:
        print(year, 'missing required data, skipping:', missing_required)
        continue

    if s1_post is None:
        print(year, 'S1 post missing, using zero S1 post patches')
    if s2_pre is None:
        print(year, 'S2 pre missing, using zero S2 patches')
    if s2_flood is None:
        print(year, 'S2 flood missing, using zero S2 patches')
    if s2_post is None:
        print(year, 'S2 post missing, using zero S2 patches')

    mask = np.load(mask_path, mmap_mode='r')
    dem = np.load(dem_path, mmap_mode='r')
    _, h, w = s1_pre.shape

    patch_dir = OUTPUT_ROOT / str(year) / 'patches'
    patch_dir.mkdir(parents=True, exist_ok=True)

    X_s1_pre, X_s1_flood, X_s1_post, X_s2_pre, X_s2_flood, X_s2_post, X_dem, Y = [], [], [], [], [], [], [], []
    meta_rows = []
    chunk_id = 0
    total_windows = 0
    kept_windows = 0
    empty_seen = 0

    for row in range(0, h - PATCH_SIZE + 1, STRIDE):
        for col in range(0, w - PATCH_SIZE + 1, STRIDE):
            total_windows += 1
            mask_patch = mask[row:row + PATCH_SIZE, col:col + PATCH_SIZE]
            flood_fraction = float(mask_patch.mean())
            if flood_fraction == 0:
                empty_seen += 1
            keep_empty = (flood_fraction == 0 and empty_seen % KEEP_EVERY_N_EMPTY == 0)
            if flood_fraction < MIN_FLOOD_FRACTION and not keep_empty:
                continue
            kept_windows += 1

            X_s1_pre.append(s1_pre[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            X_s1_flood.append(s1_flood[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            X_s1_post.append(zero_s1_patch() if s1_post is None else s1_post[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            X_s2_pre.append(zero_s2_patch() if s2_pre is None else s2_pre[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            X_s2_flood.append(zero_s2_patch() if s2_flood is None else s2_flood[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            X_s2_post.append(zero_s2_patch() if s2_post is None else s2_post[:, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            X_dem.append(dem[None, row:row + PATCH_SIZE, col:col + PATCH_SIZE])
            Y.append(mask_patch[None])
            meta_rows.append({
                'year': year,
                'chunk_id': chunk_id,
                'row': row,
                'col': col,
                'flood_fraction': flood_fraction,
                'is_empty_kept': bool(keep_empty),
            })

            if len(Y) >= CHUNK_SIZE:
                np.save(patch_dir / f'X_s1_pre_{chunk_id}.npy', np.array(X_s1_pre, dtype=np.float32))
                np.save(patch_dir / f'X_s1_flood_{chunk_id}.npy', np.array(X_s1_flood, dtype=np.float32))
                np.save(patch_dir / f'X_s1_post_{chunk_id}.npy', np.array(X_s1_post, dtype=np.float32))
                np.save(patch_dir / f'X_s2_pre_{chunk_id}.npy', np.array(X_s2_pre, dtype=np.float32))
                np.save(patch_dir / f'X_s2_flood_{chunk_id}.npy', np.array(X_s2_flood, dtype=np.float32))
                np.save(patch_dir / f'X_s2_post_{chunk_id}.npy', np.array(X_s2_post, dtype=np.float32))
                np.save(patch_dir / f'X_dem_{chunk_id}.npy', np.array(X_dem, dtype=np.float32))
                np.save(patch_dir / f'y_mask_{chunk_id}.npy', np.array(Y, dtype=np.uint8))
                pd.DataFrame(meta_rows).to_csv(patch_dir / f'patch_metadata_{chunk_id}.csv', index=False)
                patch_index.append({'year': year, 'chunk_id': chunk_id, 'patches': len(Y)})
                print('saved chunk', chunk_id, 'patches:', len(Y))
                X_s1_pre, X_s1_flood, X_s1_post, X_s2_pre, X_s2_flood, X_s2_post, X_dem, Y = [], [], [], [], [], [], [], []
                meta_rows = []
                chunk_id += 1
                gc.collect()

    if Y:
        np.save(patch_dir / f'X_s1_pre_{chunk_id}.npy', np.array(X_s1_pre, dtype=np.float32))
        np.save(patch_dir / f'X_s1_flood_{chunk_id}.npy', np.array(X_s1_flood, dtype=np.float32))
        np.save(patch_dir / f'X_s1_post_{chunk_id}.npy', np.array(X_s1_post, dtype=np.float32))
        np.save(patch_dir / f'X_s2_pre_{chunk_id}.npy', np.array(X_s2_pre, dtype=np.float32))
        np.save(patch_dir / f'X_s2_flood_{chunk_id}.npy', np.array(X_s2_flood, dtype=np.float32))
        np.save(patch_dir / f'X_s2_post_{chunk_id}.npy', np.array(X_s2_post, dtype=np.float32))
        np.save(patch_dir / f'X_dem_{chunk_id}.npy', np.array(X_dem, dtype=np.float32))
        np.save(patch_dir / f'y_mask_{chunk_id}.npy', np.array(Y, dtype=np.uint8))
        pd.DataFrame(meta_rows).to_csv(patch_dir / f'patch_metadata_{chunk_id}.csv', index=False)
        patch_index.append({'year': year, 'chunk_id': chunk_id, 'patches': len(Y)})
        print('saved final chunk', chunk_id, 'patches:', len(Y))

    print(f'{year}: candidate windows={total_windows}, kept patches={kept_windows}')

patch_index_df = pd.DataFrame(patch_index)
patch_index_df.to_csv(OUTPUT_ROOT / 'patch_index.csv', index=False)
print('\nPatch index saved:', OUTPUT_ROOT / 'patch_index.csv')
if patch_index_df.empty:
    raise RuntimeError(
        'Patch extraction produced zero chunks. Check that masks exist and are non-empty, or lower MIN_FLOOD_FRACTION / KEEP_EVERY_N_EMPTY.'
    )
display(patch_index_df.groupby('year')['patches'].sum().reset_index())


## 10. Verify Patch Outputs


In [ ]:
print('Done. Model-ready outputs saved to:', OUTPUT_ROOT)

for year in YEARS:
    patch_dir = OUTPUT_ROOT / str(year) / 'patches'
    print('\nYEAR', year)
    if not patch_dir.exists():
        print('  no patches folder')
        continue
    for f in sorted(patch_dir.glob('*.npy'))[:12]:
        arr = np.load(f, mmap_mode='r')
        print(' ', f.name, arr.shape, arr.dtype)
    meta_files = sorted(patch_dir.glob('patch_metadata_*.csv'))
    print(' metadata files:', len(meta_files))
    expected_prefixes = ['X_s1_pre', 'X_s1_flood', 'X_s1_post', 'X_s2_pre', 'X_s2_flood', 'X_s2_post', 'X_dem', 'y_mask']
    missing_prefixes = [prefix for prefix in expected_prefixes if not any(patch_dir.glob(f'{prefix}_*.npy'))]
    if missing_prefixes:
        print(' missing patch groups:', missing_prefixes)


patch_index_path = OUTPUT_ROOT / 'patch_index.csv'
if not patch_index_path.exists():
    raise FileNotFoundError(f'Missing {patch_index_path}. Run the patch extraction cell and check earlier errors.')
patch_index_df = pd.read_csv(patch_index_path)
if patch_index_df.empty:
    raise RuntimeError('patch_index.csv exists but is empty; no patch chunks were created.')
print('\nPatch totals by year:')
display(patch_index_df.groupby('year')['patches'].sum().reset_index())


## 11. Notes for Training Notebook

The patch extraction cell writes Assam-pipeline-style multi-input files under:

```text
/content/drive/MyDrive/.../Kerala_Pipeline1_Model_Ready/<year>/patches/
```

Each chunk contains:

```text
pre branch:   X_s1_pre + X_s2_pre + X_dem
flood branch: X_s1_flood + X_s2_flood + X_dem
post branch:  X_s1_post + X_s2_post + X_dem
label:        y_mask
```

If S1 post or any S2 stage is missing, the notebook writes zero patches for that input instead of skipping the year.
